[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Tree Shape &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' helpers and chosen LightGBM model, rebuilt here from the same made-up stations so that
each answer runs on its own.

In [2]:
def lightgbm_shapes(model):
    nodes = model.booster_.trees_to_dataframe()
    leaves = nodes[nodes["split_feature"].isna()]
    return leaves.groupby("tree_index").size(), leaves.groupby("tree_index")["node_depth"].max() - 1


def xgboost_shapes(model):
    trees = [text.splitlines() for text in model.get_booster().get_dump()]
    leaves = pd.Series([sum("leaf=" in line for line in lines) for lines in trees])
    depths = pd.Series([max(line.count("\t") for line in lines) for lines in trees])
    return leaves, depths


def describe(name, shapes):
    leaves, depths = shapes
    print(f"{name}: {leaves.mean():.1f} leaves on average, {leaves.max()} at most | "
          f"depth {depths.mean():.1f} on average, {depths.max()} at most")


lightgbm_chosen = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, num_leaves=8, random_state=0, verbose=-1).fit(
    X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])

**1.** Train LightGBM with `num_leaves=15`, stopped early on the validation stations at a learning rate of 0.1, and print its best iteration and validation loss beside the sweep's.

In [3]:
fifteen = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, num_leaves=15, random_state=0, verbose=-1)
fifteen.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
print(f"num_leaves 15: best {fifteen.best_iteration_} trees, validation {fifteen.best_score_['valid_0']['l2']:.3f}")

num_leaves 15: best 186 trees, validation 2.509


Fifteen leaves stop after 186 trees with a validation loss of 2.509, between the sweep's 2.352 for eight leaves and 2.546 for 31, from the same single split.

**2.** Train 100 XGBoost trees with `max_depth=4`, and print their shapes with `describe` and `xgboost_shapes`.

In [4]:
four = xgb.XGBRegressor(n_estimators=100, max_depth=4, random_state=0).fit(X_train, y_train)
describe("XGBoost, depth 4", xgboost_shapes(four))

XGBoost, depth 4: 15.8 leaves on average, 16 at most | depth 4.0 on average, 4 at most


Four levels allow 16 leaves, and XGBoost's trees hold 15.8 on average, as LightGBM's held 15.6 when capped at the same depth.

**3.** Train XGBoost leaf by leaf with `max_leaves=8` and no depth limit, stopped early at a learning rate of 0.1, and print its validation loss beside LightGBM's with `num_leaves=8`.

In [5]:
eight = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.1, grow_policy="lossguide", max_leaves=8, max_depth=0,
                         early_stopping_rounds=100, random_state=0)
eight.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
print("XGBoost, 8 leaves: ", round(mean_squared_error(y_valid, eight.predict(X_valid)), 3))
print("LightGBM, 8 leaves:", round(mean_squared_error(y_valid, lightgbm_chosen.predict(X_valid)), 3))

XGBoost, 8 leaves:  2.501
LightGBM, 8 leaves: 2.352


Grown leaf by leaf with eight leaves, XGBoost's validation loss is 2.501, against LightGBM's 2.352 with the same limit. The libraries differ in details such as XGBoost's penalty on leaf values, which the **Regularization and Sampling** notebook covers, and one split of the stations is too little to say which of them matters here.

**4.** Count how many of `lightgbm_chosen`'s trees reach seven levels, the most a tree with eight leaves can have.

In [6]:
leaves, depths = lightgbm_shapes(lightgbm_chosen)
print("trees:", len(depths), "| at depth 7:", int((depths == 7).sum()))
print("depths:", depths.value_counts().sort_index().to_dict())

trees: 493 | at depth 7: 26
depths: {3: 5, 4: 133, 5: 216, 6: 113, 7: 26}


26 of the 493 trees reach seven levels, the depth of a tree that keeps splitting one branch. Most stop at four or five levels.

---

&#8592; **Back to:** [Tree Shape](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/06-tree-shape.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
